# VLM Distillation — Results Analysis

Generates benchmark plots and training curves from `results/benchmark_results.json` and `checkpoints/history.json`.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
from pathlib import Path

plt.rcParams.update({
    'font.size': 12,
    'figure.dpi': 150,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

RESULTS_DIR = Path('../results')
CKPT_DIR    = Path('../checkpoints')
RESULTS_DIR.mkdir(exist_ok=True)

In [ ]:
# ── Accuracy numbers ──────────────────────────────────────────────────────
# Student: from checkpoints/history.json best val acc
with open(CKPT_DIR / 'history.json') as f:
    history = json.load(f)

STUDENT_ACC = max(e['acc'] for e in history['val'])
# Teacher: update after running: python eval/evaluate.py --teacher --config configs/distil_config.yaml
# The teacher head is trained as a linear probe on cached Florence-2 features.
TEACHER_ACC = 0.55  # placeholder — replace with actual output

ACCURACY_RETENTION = STUDENT_ACC / TEACHER_ACC
print(f'Student val accuracy : {STUDENT_ACC:.4f} ({STUDENT_ACC*100:.2f}%)')
print(f'Teacher acc (approx) : {TEACHER_ACC:.4f} ({TEACHER_ACC*100:.2f}%)')
print(f'Accuracy retention   : {ACCURACY_RETENTION:.1%}')

In [ ]:
# ── Load benchmark results ─────────────────────────────────────────────────
with open(RESULTS_DIR / 'benchmark_results.json') as f:
    bench = json.load(f)

df = pd.DataFrame([r for r in bench if 'error' not in r])

# Compute speedup relative to teacher FP32 @ bs=192 (max-batch baseline)
teacher_fp32_bs192 = df[
    (df['model'] == 'teacher') &
    (df['optimization'] == 'FP32 baseline') &
    (df['batch_size'] == 192)
]['throughput_samples_per_sec'].values[0]

df['speedup'] = df['throughput_samples_per_sec'] / teacher_fp32_bs192
df['accuracy_retention'] = df['model'].map({'teacher': 1.0, 'student': ACCURACY_RETENTION})

print(f'Teacher FP32 bs=192 baseline: {teacher_fp32_bs192:.1f} samples/s')
df[['model','optimization','batch_size','latency_mean_ms','throughput_samples_per_sec','speedup']].to_string(index=False)

In [ ]:
# ── Plot 1: Throughput Scaling — all batch sizes ───────────────────────────
OPT_ORDER  = ['FP32 baseline', 'bf16', 'bf16 + compile']
OPT_STYLES = {
    'FP32 baseline': dict(color='#607D8B', linestyle='-',  marker='o'),
    'bf16':          dict(color='#2196F3', linestyle='--', marker='s'),
    'bf16 + compile':dict(color='#4CAF50', linestyle='-',  marker='^'),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)

for ax, model_name in zip(axes, ['teacher', 'student']):
    sub = df[df['model'] == model_name].sort_values('batch_size')
    for opt in OPT_ORDER:
        row = sub[sub['optimization'] == opt]
        if row.empty:
            continue
        style = OPT_STYLES[opt]
        ax.plot(row['batch_size'], row['throughput_samples_per_sec'],
                label=opt, **style, linewidth=2, markersize=7)
    ax.set_xlabel('Batch Size')
    ax.set_ylabel('Throughput (samples/s)')
    ax.set_title(f'{model_name.capitalize()} Throughput Scaling')
    ax.set_xticks(df['batch_size'].unique())
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Inference Throughput vs Batch Size (H100)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'throughput_scaling.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 2: Throughput Bar Chart at bs=192 ────────────────────────────────
df192 = df[df['batch_size'] == 192].copy()

COLORS = {'teacher': '#2196F3', 'student': '#FF9800'}
labels = [f"{r['model'].capitalize()}\n{r['optimization']}" for _, r in df192.iterrows()]
throughputs = df192['throughput_samples_per_sec'].tolist()
bar_colors  = [COLORS[r['model']] for _, r in df192.iterrows()]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(labels)), throughputs, color=bar_colors, edgecolor='white', linewidth=0.5)

for bar, val, (_, row) in zip(bars, throughputs, df192.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + max(throughputs)*0.01,
            f'{val:,.0f}\n({row["speedup"]:.1f}×)',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Throughput (samples/s)')
ax.set_title('Inference Throughput — Teacher vs Student (batch_size=192, H100)', fontsize=13)
ax.grid(True, axis='y', alpha=0.3)

# Legend patches
import matplotlib.patches as mpatches
legend_patches = [mpatches.Patch(color=c, label=m.capitalize()) for m, c in COLORS.items()]
ax.legend(handles=legend_patches, loc='upper left')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'throughput_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 3: Accuracy-Retention vs Speedup scatter ─────────────────────────
df192 = df[df['batch_size'] == 192].copy()

MARKERS = {'FP32 baseline': 'o', 'bf16': 's', 'bf16 + compile': '^'}
COLORS  = {'teacher': '#2196F3', 'student': '#FF9800'}

fig, ax = plt.subplots(figsize=(8, 5))

for _, row in df192.iterrows():
    ax.scatter(
        row['speedup'], row['accuracy_retention'],
        color=COLORS[row['model']],
        marker=MARKERS.get(row['optimization'], 'o'),
        s=140, zorder=3,
        label=f"{row['model'].capitalize()} / {row['optimization']}",
    )
    ax.annotate(
        f"{row['speedup']:.1f}×",
        (row['speedup'], row['accuracy_retention']),
        textcoords='offset points', xytext=(8, 4), fontsize=9,
    )

ax.axhline(1.0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.axhline(ACCURACY_RETENTION, color='#FF9800', linestyle=':', linewidth=1, alpha=0.7)
ax.text(ax.get_xlim()[0] if ax.get_xlim()[0] > 0 else 0.5, ACCURACY_RETENTION + 0.01,
        f'Student retention ({ACCURACY_RETENTION:.0%})', color='#FF9800', fontsize=9)

ax.set_xlabel('Speedup over Teacher FP32 (bs=192)')
ax.set_ylabel('Accuracy Retention')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
ax.set_title('Accuracy Retention vs Inference Speedup')
ax.legend(loc='lower right', fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'accuracy_vs_speedup.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 4: Training Loss Curves ──────────────────────────────────────────
train_h = history['train']
val_h   = history['val']

epochs_tr = range(1, len(train_h) + 1)
epochs_val = [e['epoch'] + 1 for e in val_h]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Loss components
ax1.plot(epochs_tr, [e['loss']   for e in train_h], color='#333',       linewidth=2, label='Train total')
ax1.plot(epochs_tr, [e['l_task'] for e in train_h], color='#2196F3',    linewidth=1.5, linestyle='--', label='L_task')
ax1.plot(epochs_tr, [e['l_feat'] for e in train_h], color='#FF9800',    linewidth=1.5, linestyle=':', label='L_feat')
ax1.plot(epochs_val, [e['loss']  for e in val_h],   color='#E53935', linewidth=2, linestyle='-', marker='o', markersize=4, label='Val total')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Validation accuracy
ax2.plot(epochs_val, [e['acc'] for e in val_h], color='#4CAF50', linewidth=2, marker='o', markersize=4, label='Val accuracy')
ax2.axhline(STUDENT_ACC, color='#FF9800', linestyle='--', linewidth=1.2, label=f'Best ({STUDENT_ACC:.1%})')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('VQA Accuracy')
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
ax2.set_title('Validation Accuracy')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.suptitle('Student Distillation Training (15 epochs, VQAv2 mini-val 5k)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Summary Table ─────────────────────────────────────────────────────────
summary = df[df['batch_size'] == 192][[
    'model', 'optimization',
    'latency_mean_ms', 'latency_std_ms',
    'throughput_samples_per_sec', 'speedup'
]].copy()
summary.columns = ['Model', 'Optimization', 'Latency mean (ms)', 'Latency std (ms)', 'Throughput (samp/s)', 'Speedup']
summary['Speedup'] = summary['Speedup'].map('{:.1f}×'.format)
summary['Latency mean (ms)'] = summary['Latency mean (ms)'].map('{:.1f}'.format)
summary['Latency std (ms)']  = summary['Latency std (ms)'].map('{:.2f}'.format)
summary['Throughput (samp/s)'] = summary['Throughput (samp/s)'].map('{:,.0f}'.format)
print(summary.to_string(index=False))

print(f'\nStudent accuracy : {STUDENT_ACC:.4f} ({STUDENT_ACC*100:.2f}%)')
print(f'Teacher accuracy : {TEACHER_ACC:.4f} ({TEACHER_ACC*100:.2f}%) [linear probe on frozen Florence-2 features]')
print(f'Accuracy retention: {ACCURACY_RETENTION:.1%}')

summary.to_csv(RESULTS_DIR / 'benchmark_summary_bs192.csv', index=False)
print('\nSaved → results/benchmark_summary_bs192.csv')